# Tiền xử lý dữ liệu — Phân loại review rác tiếng Việt (ViSpamReviews v2)

**Mục đích:** notebook này biến 3 file CSV thô (`ds102_train.csv`, `ds102_dev.csv`, `ds102_test.csv`) thành **bộ đặc trưng sẵn sàng cho huấn luyện**, bám sát chiến thuật đã bàn:

- Làm sạch văn bản tiếng Việt: BOM, Unicode NFC, HTML entity, ký tự điều khiển/zero-width, xuống dòng nhúng.
- Đặc trưng thủ công (dense): thống kê bề mặt, tín hiệu spam, lexicon dịch vụ/sản phẩm, độ trùng với tên & mô tả sản phẩm, thống kê sản phẩm, comment trùng lặp.
- TF-IDF 4 khối: word + char n-gram trên `comment`, TF-IDF `product_name`, TF-IDF `description`.
- Lưu artifacts (sparse `.npz`, dense `.npy/.parquet`, `joblib`, `json`) để notebook huấn luyện chỉ cần nạp lại.

**Nguyên tắc chống rò rỉ (leakage):**
- Vectorizer và encoder **chỉ fit trên `train`**, rồi `transform` cho `dev`/`test`.
- Các thống kê cấp corpus (tần suất sản phẩm, số lần trùng comment) **không dùng nhãn**, tính trên toàn bộ corpus và áp dụng nhất quán cho cả 3 split. Có thể đổi sang chỉ `train` bằng `CONFIG["corpus_stats_on"]`.
- Không có đặc trưng nào suy ra từ `spam_label`.

**Môi trường:** pandas, numpy, scikit-learn, scipy, pyarrow, joblib. Không cần GPU, không cần torch/transformers.

## 0. Thiết lập chung

Tự động tìm thư mục gốc dự án bằng cách đi ngược lên từ thư mục hiện tại cho tới khi thấy `Data/ds102_train.csv`. Như vậy notebook chạy được cả từ `src/` lẫn từ thư mục gốc.

In [ ]:
# ============================================================
# 0. IMPORT & CẤU HÌNH
# ============================================================
import ast
import html
import json
import re
import string
import unicodedata
import warnings
from collections import Counter
from datetime import datetime
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MultiLabelBinarizer, OneHotEncoder

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 160)
pd.set_option("display.width", 180)

SEED = 42
np.random.seed(SEED)


def find_project_root(start=None, marker="Data/ds102_train.csv"):
    """Tìm thư mục gốc dự án chứa đường dẫn marker."""
    p = Path(start).resolve() if start is not None else Path.cwd().resolve()
    for cand in [p, *p.parents]:
        if (cand / marker).exists():
            return cand
    raise FileNotFoundError(f"Không tìm thấy thư mục gốc chứa '{marker}' tính từ {p}")


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "Data"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
for sub in ["clean", "dense", "sparse", "labels", "ids", "models", "meta"]:
    (ARTIFACT_DIR / sub).mkdir(parents=True, exist_ok=True)

SPLIT_FILES = {
    "train": DATA_DIR / "ds102_train.csv",
    "dev": DATA_DIR / "ds102_dev.csv",
    "test": DATA_DIR / "ds102_test.csv",
}
LABEL_COL = "spam_label"

CONFIG = {
    "version": "1.0",
    "seed": SEED,
    # "all" = train+dev+test (thống kê không nhãn) | "train" = chỉ train
    "corpus_stats_on": "all",
    "max_categories": 40,
    "tfidf": {
        "comment_word": dict(
            ngram_range=(1, 2), min_df=3, max_df=0.95, max_features=200_000,
            sublinear_tf=True, lowercase=True,
            token_pattern=r"(?u)\b\w+\b", dtype=np.float32,
        ),
        "comment_char": dict(
            analyzer="char_wb", ngram_range=(2, 5), min_df=3, max_df=0.95,
            max_features=200_000, sublinear_tf=True, lowercase=True,
            dtype=np.float32,
        ),
        "product_name": dict(
            ngram_range=(1, 2), min_df=2, max_df=0.95, max_features=50_000,
            sublinear_tf=True, lowercase=True,
            token_pattern=r"(?u)\b\w+\b", dtype=np.float32,
        ),
        "description": dict(
            ngram_range=(1, 1), min_df=5, max_df=0.90, max_features=80_000,
            sublinear_tf=True, lowercase=True,
            token_pattern=r"(?u)\b\w+\b", dtype=np.float32,
        ),
    },
}

print("PROJECT_ROOT :", PROJECT_ROOT)
print("DATA_DIR     :", DATA_DIR)
print("ARTIFACT_DIR :", ARTIFACT_DIR)
print("pandas", pd.__version__, "| numpy", np.__version__)

## 1. Đọc dữ liệu & kiểm tra nhanh

Đọc bằng `encoding="utf-8-sig"` để bỏ BOM. pandas tự xử lý `comment` chứa `\r\n` bên trong dấu ngoặc kép — **không** đếm dòng file để suy ra số bản ghi.

In [ ]:
# ============================================================
# 1. ĐỌC DỮ LIỆU
# ============================================================
raw = {}
for split, path in SPLIT_FILES.items():
    df = pd.read_csv(path, encoding="utf-8-sig")
    raw[split] = df
    print(f"{split:5s} rows={len(df):5d}  cols={len(df.columns)}")

for split in ["train", "dev"]:
    assert LABEL_COL in raw[split].columns, f"{split} thiếu cột {LABEL_COL}"
assert "id" in raw["test"].columns, "test phải có cột id"
assert list(raw["train"].columns) == list(raw["dev"].columns)

print("\nCột train/dev:", list(raw["train"].columns))
print("Cột test     :", list(raw["test"].columns))

print("\nPhân bố nhãn:")
for split in ["train", "dev"]:
    vc = raw[split][LABEL_COL].value_counts().sort_index()
    line = ", ".join(f"{k}={v} ({v / len(raw[split]):.1%})" for k, v in vc.items())
    print(f"  {split:5s}: {line}")

print("\nKiểu dữ liệu:")
print(raw["train"].dtypes.to_string())

## 2. Làm sạch văn bản

Thứ tự xử lý quan trọng:

1. `html.unescape` (2 lần cho chắc — dữ liệu crawl bị escape) → `&amp;` thành `&`.
2. `unicodedata.normalize("NFC")` — `description` gốc ở dạng NFD (tổ hợp), NFC giúp token khớp nhau.
3. Xoá ký tự zero-width / điều khiển nhưng **giữ `\n`** (vì số dòng là tín hiệu).
4. Chuẩn hoá `\r\n` → `\n`, gộp khoảng trắng.

Ngoài bản `comment_norm` (dùng để trích đặc trưng), ta tạo thêm `comment_model` cho TF-IDF: lower-case, thay URL/email/SĐT bằng token `<url>`/`<email>`/`<phone>`, gộp ký tự lặp ≥3 về 2 (giảm nhiễu mà vẫn giữ tín hiệu qua đặc trưng `repeat_char_ratio`).

In [ ]:
# ============================================================
# 2. HÀM LÀM SẠCH
# ============================================================
_WS = re.compile(r"[ \t\u00a0\u1680\u2000-\u200a\u202f\u205f\u3000]+")
_CTRL = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]")
_ZW = re.compile(r"[\u200b-\u200f\u202a-\u202e\u2060\ufeff]")
_URL = re.compile(r"(?:https?://|www\.)\S+", re.IGNORECASE)
_EMAIL = re.compile(r"[\w.+-]+@[\w-]+\.[\w.]+")
_PHONE = re.compile(r"(?<!\d)0\d{8,10}(?!\d)")
_REPEAT_RUN = re.compile(r"(.)\1{2,}")


def normalize_text(value):
    """Chuẩn hoá chung cho comment / product_name / description."""
    if not isinstance(value, str):
        return ""
    text = html.unescape(html.unescape(value))
    text = unicodedata.normalize("NFC", text)
    text = _ZW.sub(" ", text)
    text = _CTRL.sub(" ", text)
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = _WS.sub(" ", text)
    return text.strip()


def normalize_comment(value):
    """Chuẩn hoá comment nhưng giữ cấu trúc dòng."""
    text = normalize_text(value)
    text = re.sub(r"\n[ \t]*\n+", "\n", text)
    return text.strip()


def to_model_text(value):
    """Dạng text đưa vào TF-IDF: lower-case + thay thế URL/email/SĐT + gộp ký tự lặp."""
    text = normalize_comment(value).lower()
    text = _URL.sub(" <url> ", text)
    text = _EMAIL.sub(" <email> ", text)
    text = _PHONE.sub(" <phone> ", text)
    text = _REPEAT_RUN.sub(lambda m: m.group(1) * 2, text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


print("Ví dụ nhanh:")
demo = "  SHOPPPPP giao hàng cực nhanh!!!  https://tiki.vn/x  cảm ơn shop  "
print("RAW  :", repr(demo))
print("NORM :", repr(normalize_comment(demo)))
print("MODEL:", repr(to_model_text(demo)))

In [ ]:
# ============================================================
# 2.2 ÁP DỤNG LÀM SẠCH
# ============================================================
for split, df in raw.items():
    df["comment_norm"] = df["comment"].map(normalize_comment)
    df["comment_model"] = df["comment"].map(to_model_text)
    df["product_name_norm"] = df["product_name"].map(normalize_text)
    df["description_norm"] = df["description"].map(normalize_text)
    df["product_name_model"] = df["product_name_norm"].str.lower()
    df["description_model"] = df["description_norm"].str.lower()

print("Độ dài comment sau làm sạch (train):")
print(raw["train"]["comment_norm"].str.len().describe().round(1).to_string())

print("\nVí dụ theo từng nhãn (train):")
for label in sorted(raw["train"][LABEL_COL].unique()):
    sample = raw["train"].loc[raw["train"][LABEL_COL] == label, "comment_norm"].iloc[0]
    sample = sample[:180].replace("\n", " ⏎ ")
    print(f"  [{label}] {sample}")

### 2.3 Parse `categories`

Cột `categories` là **chuỗi hoá Python list** (nháy đơn), không phải JSON → dùng `ast.literal_eval`. Cột `category` (12 giá trị) dùng one-hot, còn danh sách `categories_list` dùng multi-label (top 40 token phổ biến nhất trên train).

In [ ]:
# ============================================================
# 2.3 PARSE categories
# ============================================================
def parse_categories(value):
    try:
        parsed = ast.literal_eval(value)
    except (ValueError, SyntaxError, TypeError):
        return []
    if not isinstance(parsed, (list, tuple)):
        return []
    return [normalize_text(str(item)) for item in parsed if str(item).strip()]


for split, df in raw.items():
    df["categories_list"] = df["categories"].map(parse_categories)
    df["n_categories"] = df["categories_list"].map(len)

print(raw["train"][["categories", "categories_list", "n_categories"]].head(3).to_string())
print("\nSố category khác nhau trong category:", raw["train"]["category"].nunique())
print("Số token category khác nhau:", len({t for lst in raw["train"]["categories_list"] for t in lst}))

## 3. Đặc trưng thủ công (dense)

Vì metric là **accuracy**, các tín hiệu phân biệt quan trọng nhất là:

- **Class 3 (non-review):** URL/SĐT, ký tự lặp, xuống dòng, viết hoa, ký tự không dấu.
- **Class 2 (chỉ nói thương hiệu/dịch vụ):** nhiều từ dịch vụ (`giao hàng`, `đóng gói`, `shop`, `tiki`...) nhưng ít từ về sản phẩm (`chất lượng`, `mùi`, `chất liệu`...), ít trùng với tên/mô tả sản phẩm.
- **Class 1 (fake):** văn phong quảng cáo (`tuyệt vời`, `số 1`, `nên mua`), dài bất thường, nhắc `sản phẩm` nhiều.

Các đặc trưng cấp corpus (không dùng nhãn): số lần một comment trùng lặp, số sản phẩm khác nhau mà cùng một comment xuất hiện (dấu hiệu chiến dịch spam), tần suất sản phẩm, rating trung bình của sản phẩm.

In [ ]:
# ============================================================
# 3.1 LEXICON & REGEX DÙNG CHUNG
# ============================================================
WORD_RE = re.compile(r"\w+", re.UNICODE)
EMOJI_RE = re.compile("[\U0001F300-\U0001FAFF\u2600-\u27BF\u2B00-\u2BFF\uFE0F]")
URL_RE = _URL
EMAIL_RE = _EMAIL
PHONE_RE = _PHONE
REPEAT_RUN = _REPEAT_RUN

SERVICE_WORDS = [
    "giao hàng", "vận chuyển", "ship", "đóng gói", "bao bì", "nhân viên",
    "thái độ", "phục vụ", "chăm sóc khách", "shop", "người bán", "cửa hàng",
    "tiki", "lazada", "shopee", "cảm ơn", "ủng hộ", "giao nhanh",
]
PRODUCT_WORDS = [
    "chất lượng", "sản phẩm", "mùi", "màu", "size", "kích thước", "chất liệu",
    "mẫu mã", "đường may", "công dụng", "hiệu quả", "pin", "nhựa", "kim loại",
    "vải", "gỗ", "da", "tóc", "mặt", "môi", "kem", "nước hoa", "thơm",
]
HYPE_WORDS = [
    "tuyệt vời", "hoàn hảo", "tốt nhất", "số 1", "uy tín", "nên mua",
    "đáng mua", "5 sao", "năm sao", "hài lòng", "rất tốt", "quá tốt",
    "chất lượng tốt", "yên tâm", "ủng hộ",
]
STOPWORDS = {
    "và", "của", "là", "cho", "với", "có", "không", "này", "thì", "rất",
    "mình", "tôi", "bạn", "nhưng", "nên", "được", "một", "các", "những",
    "đã", "đang", "sẽ", "khi", "để", "về", "trên", "trong", "ra", "vào",
    "rồi", "nữa", "quá", "lắm", "ạ", "nhé", "vẫn", "cũng", "đó", "nó",
}
VNI_CHARS = set("ăâđêôơưàáảãạằắẳẵặầấẩẫậèéẻẽẹềếểễệìíỉĩịòóỏõọồốổỗộờớởỡợùúủũụừứửữựỳýỷỹỵ")
VNI_CHARS |= {c.upper() for c in VNI_CHARS}

CORPUS_COLS = ["comment_dup_count", "comment_dup_products", "product_freq", "product_mean_rating"]


def tokenize(text):
    return [w for w in WORD_RE.findall(text.lower()) if w not in STOPWORDS]


print("Số từ lexicon:", len(SERVICE_WORDS), "dịch vụ |", len(PRODUCT_WORDS), "sản phẩm |", len(HYPE_WORDS), "quảng cáo")

In [ ]:
# ============================================================
# 3.2 HÀM TRÍCH ĐẶC TRƯNG
# ============================================================
def surface_features(text):
    """Thống kê bề mặt + tín hiệu spam của một comment đã chuẩn hoá."""
    n_chars = len(text)
    words = WORD_RE.findall(text)
    n_words = len(words)
    uniq_words = len({w.lower() for w in words})

    run_chars = 0
    max_run = 0
    for m in REPEAT_RUN.finditer(text):
        length = len(m.group(0))
        run_chars += length
        max_run = max(max_run, length)

    uppers = sum(ch.isupper() for ch in text)
    digits = sum(ch.isdigit() for ch in text)
    puncts = sum(ch in string.punctuation for ch in text)
    spaces = sum(ch.isspace() for ch in text)
    vni = sum(ch in VNI_CHARS for ch in text)

    return {
        "n_chars": n_chars,
        "n_words": n_words,
        "n_unique_words": uniq_words,
        "type_token_ratio": uniq_words / n_words if n_words else 0.0,
        "n_lines": text.count("\n") + 1,
        "n_sentences": len(re.findall(r"[.!?…]+", text)) + 1,
        "mean_word_len": float(np.mean([len(w) for w in words])) if words else 0.0,
        "max_word_len": max((len(w) for w in words), default=0),
        "upper_ratio": uppers / n_chars if n_chars else 0.0,
        "digit_ratio": digits / n_chars if n_chars else 0.0,
        "punct_ratio": puncts / n_chars if n_chars else 0.0,
        "space_ratio": spaces / n_chars if n_chars else 0.0,
        "vni_ratio": vni / n_chars if n_chars else 0.0,
        "n_emoji": len(EMOJI_RE.findall(text)),
        "n_exclaim": text.count("!"),
        "n_question": text.count("?"),
        "repeat_char_ratio": run_chars / n_chars if n_chars else 0.0,
        "max_repeat_run": max_run,
        "has_url": int(bool(URL_RE.search(text))),
        "has_email": int(bool(EMAIL_RE.search(text))),
        "has_phone": int(bool(PHONE_RE.search(text))),
        "n_upper_words": sum(1 for w in words if len(w) >= 2 and w.isupper()),
    }


def lexicon_features(text_lower):
    """Mức độ xuất hiện của từ vựng dịch vụ / sản phẩm / quảng cáo."""
    service = sum(text_lower.count(w) for w in SERVICE_WORDS)
    product = sum(text_lower.count(w) for w in PRODUCT_WORDS)
    hype = sum(text_lower.count(w) for w in HYPE_WORDS)
    n_words = max(len(WORD_RE.findall(text_lower)), 1)
    return {
        "service_hits": service,
        "product_hits": product,
        "hype_hits": hype,
        "service_ratio": service / n_words,
        "product_ratio": product / n_words,
        "hype_ratio": hype / n_words,
        "service_minus_product": service - product,
        "service_over_product": service / (product + 1.0),
    }


_DESC_TOKEN_CACHE = {}
_PROD_TOKEN_CACHE = {}


def _cached_tokens(text, cache):
    if text not in cache:
        cache[text] = set(tokenize(text))
    return cache[text]


def overlap_features(comment, product_name, description):
    """Độ trùng từ vựng giữa comment và tên/mô tả sản phẩm (bắt class 3 lạc đề)."""
    a = set(tokenize(comment))
    p = _cached_tokens(product_name, _PROD_TOKEN_CACHE)
    d = _cached_tokens(description, _DESC_TOKEN_CACHE)

    def jaccard(x, y):
        union = x | y
        return len(x & y) / len(union) if union else 0.0

    def coverage(x, y):
        return len(x & y) / len(y) if y else 0.0

    return {
        "jac_comment_product": jaccard(a, p),
        "cover_product_in_comment": coverage(a, p),
        "jac_comment_description": jaccard(a, d),
        "cover_description_in_comment": coverage(a, d),
        "n_tokens_comment": len(a),
    }


def metadata_features(df):
    """Đặc trưng metadata + độ dài văn bản phụ."""
    out = pd.DataFrame(index=df.index)
    out["rating"] = df["rating"].astype(float)
    out["is_5_star"] = (df["rating"] == 5).astype(int)
    out["is_low_star"] = (df["rating"] <= 2).astype(int)
    out["log_num_sold"] = np.log1p(df["num_sold"].astype(float))
    out["log_num_review"] = np.log1p(df["num_review"].astype(float))
    review_per_sold = df["num_review"].astype(float) / (df["num_sold"].astype(float) + 1.0)
    out["log_review_per_sold"] = np.log1p(review_per_sold)
    out["no_sold"] = (df["num_sold"] == 0).astype(int)
    out["no_review"] = (df["num_review"] == 0).astype(int)
    out["n_categories"] = df["n_categories"].astype(float)
    out["product_name_len"] = df["product_name_norm"].str.len().astype(float)
    out["product_name_words"] = df["product_name_norm"].map(
        lambda s: len(WORD_RE.findall(s))
    ).astype(float)
    out["description_len"] = df["description_norm"].str.len().astype(float)
    out["description_words"] = df["description_norm"].map(
        lambda s: len(WORD_RE.findall(s))
    ).astype(float)
    return out


def build_dense_features(df):
    """Trả về DataFrame dense theo thứ tự cột cố định."""
    records = []
    for text, pname, desc in zip(
        df["comment_norm"], df["product_name_norm"], df["description_norm"]
    ):
        rec = {}
        rec.update(surface_features(text))
        rec.update(lexicon_features(text.lower()))
        rec.update(overlap_features(text, pname, desc))
        records.append(rec)
    surface = pd.DataFrame(records, index=df.index)
    meta = metadata_features(df)
    corpus = df[CORPUS_COLS].astype(float)
    return pd.concat([surface, meta, corpus], axis=1)


print("Đã định nghĩa xong các hàm đặc trưng.")

In [ ]:
# ============================================================
# 3.3 THỐNG KÊ CẤP CORPUS (KHÔNG DÙNG NHÃN)
# ============================================================
def add_corpus_features(frames, config):
    """Thêm đặc trưng thống kê toàn corpus, áp dụng nhất quán cho cả 3 split."""
    pieces = []
    for split, df in frames.items():
        tmp = df[["comment_norm", "product_name_norm", "rating"]].copy()
        tmp["__split__"] = split
        pieces.append(tmp)
    combo = pd.concat(pieces, ignore_index=True)

    base = combo if config["corpus_stats_on"] == "all" else combo[combo["__split__"] == "train"]

    dup_count = base["comment_norm"].value_counts()
    combo["comment_dup_count"] = combo["comment_norm"].map(dup_count).fillna(0).astype(int)

    dup_products = base.groupby("comment_norm")["product_name_norm"].nunique()
    combo["comment_dup_products"] = (
        combo["comment_norm"].map(dup_products).fillna(0).astype(int)
    )

    prod_freq = base["product_name_norm"].value_counts()
    combo["product_freq"] = combo["product_name_norm"].map(prod_freq).fillna(0).astype(int)

    prod_rating = base.groupby("product_name_norm")["rating"].mean()
    combo["product_mean_rating"] = (
        combo["product_name_norm"].map(prod_rating).fillna(base["rating"].mean()).astype(float)
    )

    for split in frames:
        mask = combo["__split__"] == split
        idx = combo.index[mask]
        for col in CORPUS_COLS:
            frames[split][col] = combo.loc[idx, col].to_numpy()
    return frames


raw = add_corpus_features(raw, CONFIG)

print("Đặc trưng cấp corpus (train):")
print(raw["train"][CORPUS_COLS].describe().round(3).to_string())
print("\nComment trùng lặp nhiều nhất (train):")
print(raw["train"].nlargest(3, "comment_dup_count")[["comment_dup_count", "comment_dup_products", "comment_norm"]].to_string())

In [ ]:
# ============================================================
# 3.4 TẠO BẢNG ĐẶC TRƯNG DENSE CHO TỪNG SPLIT
# ============================================================
dense_frames = {}
for split, df in raw.items():
    dense_frames[split] = build_dense_features(df)
    print(f"{split:5s} dense shape = {dense_frames[split].shape}")

DENSE_FEATURE_NAMES = list(dense_frames["train"].columns)
for split, frame in dense_frames.items():
    assert list(frame.columns) == DENSE_FEATURE_NAMES, f"Thứ tự cột lệch ở {split}"

# Làm sạch NaN/inf phòng thủ (các phép chia đã được bảo vệ, nhưng vẫn kiểm tra)
for split, frame in dense_frames.items():
    arr = frame.to_numpy(dtype=np.float64)
    bad = ~np.isfinite(arr)
    if bad.any():
        print(f"CẢNH BÁO: {split} có {bad.sum()} giá trị NaN/inf — thay bằng 0")
        arr[bad] = 0.0
        dense_frames[split] = pd.DataFrame(arr, columns=DENSE_FEATURE_NAMES, index=frame.index)

print("\nSố đặc trưng dense:", len(DENSE_FEATURE_NAMES))
print(DENSE_FEATURE_NAMES)

### 3.5 Báo cáo đặc trưng

Bảng mô tả minh hoạ mức phân biệt của từng đặc trưng theo nhãn trên `train`. Chỉ để tham khảo — mô hình sẽ tự học trọng số.

In [ ]:
# ============================================================
# 3.5 BÁO CÁO ĐẶC TRƯNG THEO NHÃN (chỉ để tham khảo)
# ============================================================
report = dense_frames["train"].copy()
report[LABEL_COL] = raw["train"][LABEL_COL].to_numpy()
by_label = report.groupby(LABEL_COL).mean().T
by_label.columns = [f"mean_label_{c}" for c in by_label.columns]
summary = dense_frames["train"].describe().T[["mean", "std", "min", "max"]]
feature_report = summary.join(by_label)

print("Top đặc trưng phân biệt class 3 so với class 0 (chênh lệch trung bình):")
diff3 = (by_label.get("mean_label_3", 0) - by_label.get("mean_label_0", 0)).sort_values(ascending=False)
print(diff3.head(8).round(4).to_string())
print("\nTop đặc trưng phân biệt class 2 so với class 0:")
diff2 = (by_label.get("mean_label_2", 0) - by_label.get("mean_label_0", 0)).sort_values(ascending=False)
print(diff2.head(8).round(4).to_string())

## 4. TF-IDF

4 khối vector hoá, **fit trên `train`** rồi transform cho cả 3 split:

| Khối | Cột đầu vào | Cấu hình chính |
|---|---|---|
| `comment_word` | `comment_model` | word 1–2 gram, `min_df=3`, 200k features |
| `comment_char` | `comment_model` | char_wb 2–5 gram, `min_df=3`, 200k features |
| `product_name` | `product_name_model` | word 1–2 gram, `min_df=2`, 50k features |
| `description` | `description_model` | word 1 gram, `min_df=5`, 80k features |

`sublinear_tf=True` và `dtype=float32` để giảm dung lượng. Không bỏ dấu tiếng Việt — dấu là tín hiệu ngôn ngữ quan trọng.

In [ ]:
# ============================================================
# 4.1 FIT TF-IDF TRÊN TRAIN
# ============================================================
TEXT_COLUMNS = {
    "comment_word": "comment_model",
    "comment_char": "comment_model",
    "product_name": "product_name_model",
    "description": "description_model",
}

vectorizers = {}
for name, params in CONFIG["tfidf"].items():
    column = TEXT_COLUMNS[name]
    vec = TfidfVectorizer(**params)
    vec.fit(raw["train"][column].fillna(""))
    vectorizers[name] = vec
    print(f"{name:14s} | vocab={len(vec.vocabulary_):>7,} | input={column}")

print("\nĐã fit xong 4 vectorizer trên train.")

### 4.2 Encoder cho `category` và `categories`

- `category` (12 giá trị) → one-hot, `handle_unknown="ignore"`.
- `categories_list` → multi-label nhị phân, giới hạn top 40 token phổ biến nhất trên train để tránh chiều quá lớn.

In [ ]:
# ============================================================
# 4.2 ENCODER category / categories
# ============================================================
category_ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=True, dtype=np.float32)
category_ohe.fit(raw["train"][["category"]])

counter = Counter(tok for lst in raw["train"]["categories_list"] for tok in lst)
top_categories = [tok for tok, _ in counter.most_common(CONFIG["max_categories"])]
categories_mlb = MultiLabelBinarizer(classes=top_categories, sparse_output=True)
categories_mlb.fit(raw["train"]["categories_list"])

print("One-hot category:", category_ohe.get_feature_names_out().tolist())
print("Số token categories giữ lại:", len(top_categories))
print("Top 10:", top_categories[:10])

In [ ]:
# ============================================================
# 4.3 TRANSFORM TOÀN BỘ SPLIT
# ============================================================
blocks = {}
for split, df in raw.items():
    for name, vec in vectorizers.items():
        column = TEXT_COLUMNS[name]
        blocks[(split, name)] = vec.transform(df[column].fillna("")).astype(np.float32)
    print(f"{split:5s} đã transform xong")

print("\nKích thước các khối TF-IDF (train):")
for name in vectorizers:
    mat = blocks[("train", name)]
    print(f"  {name:14s} shape={mat.shape}  nnz={mat.nnz:,}")

## 5. Ghép đặc trưng & lưu artifacts

Thứ tự khối cố định — **phải giữ nguyên** ở notebook huấn luyện:

1. `dense` (đặc trưng thủ công)
2. `category_onehot`
3. `categories_multilabel`
4. `tfidf_comment_word`
5. `tfidf_comment_char`
6. `tfidf_product_name`
7. `tfidf_description`

Lưu:
- `artifacts/sparse/X_{split}.npz` — ma trận CSR float32
- `artifacts/dense/dense_{split}.npy` + `.parquet` — đặc trưng dense thô
- `artifacts/labels/y_{split}.npy` — nhãn
- `artifacts/ids/test_ids.csv` — ID test đúng thứ tự
- `artifacts/clean/clean_{split}.parquet` — dữ liệu đã làm sạch
- `artifacts/models/preprocessors.joblib` — vectorizer + encoder đã fit
- `artifacts/meta/preprocess_config.json` — cấu hình, shape, block offsets

In [ ]:
# ============================================================
# 5.1 GHÉP MA TRẬN
# ============================================================
X = {}
dense_arrays = {}
block_names = ["dense", "category_onehot", "categories_multilabel"] + [
    f"tfidf_{name}" for name in vectorizers
]

for split, df in raw.items():
    dense_arr = dense_frames[split].to_numpy(dtype=np.float32)
    dense_arrays[split] = dense_arr

    parts = [
        sparse.csr_matrix(dense_arr),
        category_ohe.transform(df[["category"]]).astype(np.float32),
        categories_mlb.transform(df["categories_list"]).astype(np.float32),
    ]
    parts += [blocks[(split, name)] for name in vectorizers]

    X[split] = sparse.hstack(parts, format="csr")
    print(f"{split:5s} X shape={X[split].shape}  nnz={X[split].nnz:,}  "
          f"density={X[split].nnz / (X[split].shape[0] * X[split].shape[1]):.5f}")

# Kiểm tra nhất quán số cột
n_features = X["train"].shape[1]
assert X["dev"].shape[1] == n_features and X["test"].shape[1] == n_features

# Thông tin block
widths = [dense_arrays["train"].shape[1],
          category_ohe.transform(raw["train"][["category"]]).shape[1],
          categories_mlb.transform(raw["train"]["categories_list"]).shape[1]]
widths += [blocks[("train", name)].shape[1] for name in vectorizers]
assert sum(widths) == n_features, "Tổng độ rộng block không khớp số cột"

block_info = []
offset = 0
for name, width in zip(block_names, widths):
    block_info.append({"name": name, "start": offset, "end": offset + width, "width": int(width)})
    offset += width

print("\nCác khối đặc trưng:")
for info in block_info:
    print(f"  {info['name']:24s} [{info['start']:>7,} : {info['end']:>7,})  width={info['width']:,}")
print("Tổng:", n_features)

In [ ]:
# ============================================================
# 5.2 LƯU ARTIFACTS
# ============================================================
for split in raw:
    sparse.save_npz(ARTIFACT_DIR / "sparse" / f"X_{split}.npz", X[split], compressed=False)
    np.save(ARTIFACT_DIR / "dense" / f"dense_{split}.npy", dense_arrays[split])

    clean_cols = [
        "comment", "comment_norm", "comment_model",
        "product_name", "product_name_norm", "product_name_model",
        "description", "description_norm", "description_model",
        "category", "rating", "num_sold", "num_review", "n_categories",
    ]
    clean_df = raw[split][clean_cols].copy()
    clean_df["categories_list"] = raw[split]["categories_list"].map(
        lambda lst: json.dumps(lst, ensure_ascii=False)
    )
    for col in CORPUS_COLS:
        clean_df[col] = raw[split][col].to_numpy()
    if LABEL_COL in raw[split].columns:
        clean_df[LABEL_COL] = raw[split][LABEL_COL].to_numpy()
    if "id" in raw[split].columns:
        clean_df["id"] = raw[split]["id"].to_numpy()
    clean_df.to_parquet(ARTIFACT_DIR / "clean" / f"clean_{split}.parquet", index=False)

for split in ["train", "dev"]:
    np.save(ARTIFACT_DIR / "labels" / f"y_{split}.npy",
            raw[split][LABEL_COL].to_numpy(dtype=np.int64))

raw["test"][["id"]].rename(columns={"id": "ID"}).to_csv(
    ARTIFACT_DIR / "ids" / "test_ids.csv", index=False, encoding="utf-8-sig"
)

joblib.dump({
    "config": CONFIG,
    "vectorizers": vectorizers,
    "category_ohe": category_ohe,
    "categories_mlb": categories_mlb,
    "top_categories": top_categories,
    "dense_feature_names": DENSE_FEATURE_NAMES,
    "text_columns": TEXT_COLUMNS,
    "block_info": block_info,
    "n_features": int(n_features),
}, ARTIFACT_DIR / "models" / "preprocessors.joblib", compress=3)

meta = {
    "version": CONFIG["version"],
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "rows": {split: int(len(df)) for split, df in raw.items()},
    "n_features": int(n_features),
    "tfidf_vocab": {name: int(len(vec.vocabulary_)) for name, vec in vectorizers.items()},
    "blocks": block_info,
    "dense_features": DENSE_FEATURE_NAMES,
    "corpus_stats_on": CONFIG["corpus_stats_on"],
    "config": CONFIG,
}
with open(ARTIFACT_DIR / "meta" / "preprocess_config.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2, default=str)

print("Đã lưu artifacts vào:", ARTIFACT_DIR)
for sub in ["sparse", "dense", "labels", "ids", "clean", "models", "meta"]:
    files = sorted((ARTIFACT_DIR / sub).glob("*"))
    print(f"\n[{sub}]")
    for path in files:
        print(f"  {path.name:32s} {path.stat().st_size / 1e6:8.2f} MB")

## 6. Kiểm tra cuối

Các assert dưới đây đảm bảo:
- Số dòng khớp với CSV gốc.
- Số cột giống nhau giữa 3 split.
- Không NaN/inf trong dense.
- Nhãn nằm trong `{0,1,2,3}`.
- Thứ tự ID test đúng `ds102_0 … ds102_3973`.

In [ ]:
# ============================================================
# 6. KIỂM TRA TÍNH ĐÚNG ĐẮN
# ============================================================
for split in raw:
    assert X[split].shape[0] == len(raw[split]), f"{split}: số dòng không khớp"
    assert X[split].shape[1] == n_features, f"{split}: số cột không khớp"
    assert np.isfinite(dense_arrays[split]).all(), f"{split}: dense có NaN/inf"

for split in ["train", "dev"]:
    y = raw[split][LABEL_COL].to_numpy()
    assert set(np.unique(y)).issubset({0, 1, 2, 3}), f"{split}: nhãn lạ"

test_ids = raw["test"]["id"].tolist()
assert test_ids == [f"ds102_{i}" for i in range(len(test_ids))], "ID test không đúng thứ tự"

print("TẤT CẢ KIỂM TRA ĐỀU ĐẠT\n")
print("Tóm tắt:")
print(f"  train: X={X['train'].shape}, y={len(raw['train'])}")
print(f"  dev  : X={X['dev'].shape}, y={len(raw['dev'])}")
print(f"  test : X={X['test'].shape}")
print(f"  nnz  : train={X['train'].nnz:,} | dev={X['dev'].nnz:,} | test={X['test'].nnz:,}")
print(f"  dense features: {len(DENSE_FEATURE_NAMES)}")
print(f"  tổng đặc trưng: {n_features:,}")

print("\nVí dụ 5 comment theo nhãn (train):")
for label in sorted(raw["train"][LABEL_COL].unique()):
    es = raw["train"].loc[raw["train"][LABEL_COL] == label, "comment_norm"]
    print(f"  nhãn {label} (n={len(es)}): {es.iloc[0][:140]}")

## 7. Cách dùng artifacts ở notebook huấn luyện

```python
import joblib
import numpy as np
from scipy import sparse

X_train = sparse.load_npz("artifacts/sparse/X_train.npz")
X_dev   = sparse.load_npz("artifacts/sparse/X_dev.npz")
X_test  = sparse.load_npz("artifacts/sparse/X_test.npz")
y_train = np.load("artifacts/labels/y_train.npy")
y_dev   = np.load("artifacts/labels/y_dev.npy")

pre = joblib.load("artifacts/models/preprocessors.joblib")
# pre["dense_feature_names"], pre["block_info"], pre["vectorizers"], ...
```

**Lưu ý khi huấn luyện:**

- `dense` nằm ở đầu ma trận; các mô hình tuyến tính nên scale riêng khối dense (ví dụ `StandardScaler(with_mean=False)` chỉ áp phần dense) — TF-IDF đã chuẩn hoá L2 nên không cần scale.
- `CONFIG["corpus_stats_on"] = "all"` dùng thống kê không nhãn trên toàn corpus. Nếu muốn tuyệt đối "chỉ train", đổi sang `"train"` và chạy lại notebook.
- Với submission cuối cùng, có thể fit lại vectorizer trên `train + dev` rồi transform `test`; khi đó dùng `preprocessors.joblib` làm tham chiếu cấu hình.
- Notebook này chỉ tiền xử lý — chưa huấn luyện mô hình. Bước tiếp theo: Logistic Regression / LinearSVC / ComplementNB / LightGBM + soft-voting trên `dev`, tối ưu **accuracy**.